# BASİT GÖREV PLANLAYAN AI AGENT (MİNİ PROJE 2)
**Amaç:** İnsan karar mekanizmasını taklit ederek günlük görevleri aciliyet, önem ve sürelerine göre sıralayan ve kullanıcıdan görev alabilen bir Yapay Zeka Ajanı (MLP) geliştirmek.

### Bu Hücre Ne Yapıyor?
Bu kod hücresi, projenin çalışması için gereken temel veri bilimi ve makine öğrenmesi kütüphanelerini (Pandas, Numpy, Scikit-Learn) sisteme dahil eder.

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from IPython.display import display

### Bu Hücre Ne Yapıyor?
Bu hücre, ajanın öğrenebilmesi için 500 satırlık görev veri seti oluşturur. 
Görevlerin aciliyet durumu (acil: 1 veya acil değil: 0), önem derecesi (1-5) ve süresi (5-300 dk) rastgele belirlenir. Ajanın öğrenmesi hedeflenen mantık şudur: Eğer bir görev acil ise veya önemi 4'ten büyük ve 1 saatten kısa sürecek ise bu görev "öncelikli (1)" olarak etiketlenir.

In [2]:
# Kod her çalıştırıldığında sonuçların aynı çıkması için rastgelelik tohumu sabitleniyor.
np.random.seed(42)
veri_sayisi = 500

# Aciliyet: 1 (Acil), 0 (Acil Değil) | Önem: 1-5 arası | Süre: 5-300 dakika arası
aciliyet = np.random.choice([0, 1], veri_sayisi)
onem = np.random.randint(1, 6, veri_sayisi) 
sure = np.random.randint(5, 301, veri_sayisi) 

# Öncelik Belirleme
oncelik = []
for a, o, s in zip(aciliyet, onem, sure):
    # Eğer görev acil ise veya önem derecesi 4-5 arası ve 60 dakikadan kısa sürüyor ise önceliklidir
    if a == 1 or (o >= 4 and s <= 60):
        oncelik.append(1) # Öncelikli (Yapılmalı)
    else:
        oncelik.append(0) # Ertelenebilir

# Verileri Pandas DataFrame formatına dönüştürme
df = pd.DataFrame({
    'Aciliyet': aciliyet, 
    'Onem_Derecesi': onem, 
    'Sure_Dakika': sure, 
    'Oncelik_Hedefi': oncelik
})

print("Veri seti başarıyla oluşturuldu!")

Veri seti başarıyla oluşturuldu!


### Bu Hücre Ne Yapıyor?
Bu hücrede veriler makine öğrenmesi modeline girmeden önce hazırlanmaktadır.
1. Veri seti %80 Eğitim (Train) ve %20 Test olarak ikiye ayrılır.
2. `StandardScaler` kullanılarak değerler standartlaştırılır yani aynı ölçeğe getirilir. Bu sayede "süre" gibi büyük değerli değişkenlerin, "aciliyet" gibi küçük değerli değişkenleri baskılaması engellenir.

In [3]:
# x ve y değişkenlerini ayırma
X = df[['Aciliyet', 'Onem_Derecesi', 'Sure_Dakika']]
y = df['Oncelik_Hedefi']

# %80 Eğitim (Train) ve %20 Test olarak ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modeli yanıltmaması için verileri aynı ölçeğe getirme (Standardizasyon)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Veriler eğitim için bölündü ve ölçeklendirildi!")

Veriler eğitim için bölündü ve ölçeklendirildi!


### Bu Hücre Ne Yapıyor?
Bu hücrede ajanın karar vermesini sağlayan Çok Katmanlı Algılayıcı (MLP - Yapay Sinir Ağı) mimarisi kurulur ve eğitilir. Modelde 16 ve 8 nörondan oluşan iki gizli katman (hidden layer) kullanılmış, eğitim sonucunda modelin test verisi üzerindeki başarı oranı ekrana yazdırılmıştır.

In [4]:
print("Ajan eğitiliyor...\n")

# 2 Gizli Katman (16 ve 8 nöron)
model = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', solver='adam', max_iter=2000, random_state=42)

# Modeli eğitim verileriyle eğitme
model.fit(X_train_scaled, y_train)

# Eğitim sonrası test verisiyle modelin başarısını ölçme
test_basarisi = model.score(X_test_scaled, y_test)
print(f" Eğitim tamamlandı! Modelin Test Başarısı: %{test_basarisi * 100:.2f}")

Ajan eğitiliyor...

 Eğitim tamamlandı! Modelin Test Başarısı: %99.00


### Bu Hücre Ne Yapıyor?
Eğitilen ajanın gerçek dünya performansını test etmek için tasarlanmış bir modüldür. 
Bu hücre çalıştırıldığında bir döngü (`while True`) başlatılarak kullanıcıdan yeni görevler (görev adı, aciliyeti, önemi ve süresi) istenir. Kullanıcı "bitti" yazdığında döngü sonlanır, girilen görevler modelin anlayacağı ölçeğe (`scaler.transform`) aktarılır ve ajan tarafından önceliklendirilir ve sonuç tablosu ekrana yazdırılır.

In [ ]:
print("\n--- GÖREV PLANLAMA ---")
print("Lütfen görevlerinizi tek tek girin. Planlamayı başlatmak için Görev Adına 'bitti' yazın.\n")

adlar = []
aciliyetler = []
onemler = []
sureler = []

# Kullanıcıdan dinamik olarak görevleri alan döngü
while True:
    g_adi = input("Görev Adı: ")
    
    if g_adi.lower() == 'bitti':
        break
        
    try:
        g_acil = int(input("Aciliyet (1: Acil, 0: Acil Değil): "))
        g_onem = int(input("Önem Derecesi (1-5 arası): "))
        g_sure = int(input("Süre (Dakika cinsinden): "))
        
        adlar.append(g_adi)
        aciliyetler.append(g_acil)
        onemler.append(g_onem)
        sureler.append(g_sure)
        print("Görev listeye eklendi!\n")
    except ValueError:
        print("Hata: Aciliyet, önem ve süre için sadece rakam kullanılmalıdır! Lütfen görevi tekrar girin.\n")

# Eğer listeye en az bir görev eklendiyse planlamayı yap
if len(adlar) > 0:
    yeni_gorevler = pd.DataFrame({
        'Gorev_Adi': adlar,
        'Aciliyet': aciliyetler,
        'Onem_Derecesi': onemler,
        'Sure_Dakika': sureler
    })

    # Yeni görevler eğitimde yapıldığı gibi aynı ölçeğe getiriliyor.
    gorevler_scaled = scaler.transform(yeni_gorevler[['Aciliyet', 'Onem_Derecesi', 'Sure_Dakika']])

    # predict_proba ile %0 ile %100 arasında bir öncelik olasılık skoru hesaplatılıyor.
    tahmin_skorlari = model.predict_proba(gorevler_scaled)[:, 1]

    # Skorları tabloya yansıtma ve en yüksek puana göre sıralama
    yeni_gorevler['Yapay_Zeka_Skoru (%)'] = (tahmin_skorlari * 100).round(2)
    siralanmis_plan = yeni_gorevler.sort_values(by='Yapay_Zeka_Skoru (%)', ascending=False)

    print("\n Ajan Görevlerinizi Analiz Etti! İşte Çalışma Planınız: \n")
    display(siralanmis_plan)
else:
    print("\n Hiç görev girmediniz, planlama yapılamadı.")


--- GÖREV PLANLAMA ---
Lütfen görevlerinizi tek tek girin. Planlamayı başlatmak için Görev Adına 'bitti' yazın.

